# Copy documents from one collection to another

This example explores how index item can be copy to a new/other collection.


### Access required

The content of this notebook requires access to Deep Search capabilities which are not
available on the public access system.

[Contact us](https://ds4sd.github.io) if you are interested in exploring
these Deep Search capabilities.

### Set notebooks parameters

In [1]:
from dsnotebooks.settings import CopyCollDocumentNotebookSettings

# notebook settings auto-loaded from .env / env vars
notebook_settings = CopyCollDocumentNotebookSettings()

PROFILE_NAME = notebook_settings.profile  # profile to use
PROJ_KEY = notebook_settings.proj_key  # project to use
INDEX_KEY = notebook_settings.index_key  # index to use
NEW_INDEX_NAME = notebook_settings.new_idx_name  # new index to use
DOC_NAME = notebook_settings.document_name  # document to copy
CLEANUP = notebook_settings.cleanup  # whether to clean up

WAIT_S = 3

### Connect to Deep Search

In [2]:
from typing import Optional

import deepsearch as ds
from deepsearch.cps.client.components.data_indices import (
    DataIndexItemUrls,
    ElasticProjectDataCollectionSource,
)

api = ds.CpsApi.from_env(profile_name=PROFILE_NAME)

### List collections in project

In [3]:
collections = api.data_indices.list(proj_key=PROJ_KEY)

for dataindex in collections:
    print(f"index_name: {dataindex.name}, index_key: {dataindex.source.index_key}")

index_name: 208afeb_3595e30-ocr, index_key: a5bc1ca67923f9934a51f86315a1aa5e7e7af5a2
index_name: 208afeb_3595e30-raw, index_key: db5d279cf71f3e61b0bdfd5c85adafe592cdb7f9
index_name: 244fa8d_53dcc8f, index_key: d5768ecd3c9caf4e39291ba12fa0a943c0a2146c
index_name: 38c7afb_eaeb165-ocr, index_key: d0c1553e82a9eb6f22acc47d255187328ab9034b
index_name: 38c7afb_eaeb165-raw, index_key: 9713088d370c4fde8376e2d570a92a48b1433b60
index_name: 40a7236_317d2b4-ocr, index_key: 69431c668920c6e16a921ef16a5adefc0da5108f
index_name: 6dcfb01_5ab483a, index_key: 30e01985ff09b0f4f9c9ffe0511a0cf694cdefac
index_name: 9a765ea_e2fc9b3, index_key: b5adb77a5471f75fa7f6b2de33a6b411069e61b9
index_name: 9a765ea_e2fc9b3-ocr, index_key: 9391aa17f5b7ba5609e23767b24460a45e9aba71
index_name: 9a765ea_e2fc9b3-raw, index_key: 1ff725e720bf2b6ebe4ff45bccddfd60616998db
index_name: 9b8ea5e_8006e4a-ocr, index_key: 499f25c5e736e3bd7ae481f464205fbc4d57d950
index_name: Annual Reports, index_key: d82c8a03126801837a459e8e26601cb5edfe00

### Get document from collection

In [4]:
indices = api.data_indices.list(proj_key=PROJ_KEY)

dataindex = next((x for x in indices if x.source.index_key == INDEX_KEY), None)

search_query = dataindex.list_items(api, DOC_NAME)

doc_id: Optional[str] = None
for item in search_query:
    doc_id = item["id"]

if not doc_id:
    print("No document found")
else:
    print(f"doc_id: {doc_id}")

doc_id: 6627d1b67955c51ff1aa8858de671bb5a62ad70c77e62e0ac57c153d0078b7ea


## Target collection

#### Option 1:  Create New Collection
- Fill new_collection_name

In [ ]:
new_data_index = api.data_indices.create(
    proj_key=PROJ_KEY, name=NEW_INDEX_NAME, type="deepsearch-doc"
)

new_index_key = new_data_index.source.index_key
print(new_index_key)

#### Option 2: Fill index key (collection already created) from previous list of collections

In [ ]:
# new_index_key = ""

### Get document urls

In [ ]:
item: DataIndexItemUrls = dataindex.get_item_urls(api, doc_id)

pdf_url = item.pdf_url
print(pdf_url)

### Upload document to target collection

In [ ]:
from deepsearch.cps.data_indices import utils as data_indices_utils

coords = ElasticProjectDataCollectionSource(proj_key=PROJ_KEY, index_key=new_index_key)

data_indices_utils.upload_files(api=api, coords=coords, url=pdf_url)

### Cleanup

In [ ]:
if CLEANUP:
    api.data_indices.delete(coords)